In [ ]:
!pip install -q transformers accelerate bitsandbytes datasets pandas matplotlib
!pip install -q autoawq
!pip install -q llama-cpp-python
!pip install -q huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 MB 12.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.4 MB/s eta 0:00:00


In [ ]:
import torch, time, gc, json, os
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))

CUDA available: True
GPU: Tesla T4
Compute capability: (7, 5)


In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 19.7 MB/s eta 0:00:00


In [ ]:
import bitsandbytes
print(bitsandbytes.__version__)

0.50.0


In [ ]:
from datasets import load_dataset
import random

random.seed(42)


gsm8k = load_dataset("openai/gsm8k", "main", split="test")
gsm8k = gsm8k.shuffle(seed=42).select(range(200))

def extract_gsm8k_answer(answer_field):

    return answer_field.split("####")[-1].strip().replace(",", "")

gsm8k_questions = [
    {"question": ex["question"], "gold": extract_gsm8k_answer(ex["answer"])}
    for ex in gsm8k
]
print(f"GSM8K subset: {len(gsm8k_questions)} questions")
print(gsm8k_questions[0])


mmlu = load_dataset("cais/mmlu", "all", split="test")
mmlu = mmlu.shuffle(seed=42).select(range(200))

letters = ["A", "B", "C", "D"]
mmlu_questions = []
for ex in mmlu:
    choices_str = "\n".join(f"{letters[i]}. {c}" for i, c in enumerate(ex["choices"]))
    mmlu_questions.append({
        "question": f"{ex['question']}\n{choices_str}",
        "gold": letters[ex["answer"]],
        "subject": ex["subject"],
    })
print(f"MMLU subset: {len(mmlu_questions)} questions")
print(mmlu_questions[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


GSM8K subset: 200 questions
{'question': "Darrell and Allen's ages are in the ratio of 7:11. If their total age now is 162, calculate Allen's age 10 years from now.", 'gold': '109'}
MMLU subset: 200 questions
{'question': 'Positronium is an atom formed by an electron and a positron (antielectron). It is similar to the hydrogen atom, with the positron replacing the proton. If a positronium atom makes a transition from the state with n=3 to a state with n=1, the energy of the photon emitted in this transition is closest to\nA. 6.0 e\nB. 6.8 eV\nC. 12.2 eV\nD. 13.6 eV', 'gold': 'A', 'subject': 'college_physics'}


In [ ]:
import re
import torch

GSM8K_PROMPT = (
    "Solve this math problem step by step, then give the final numeric answer "
    "on the last line as 'Answer: <number>'.\n\nProblem: {q}\n"
)
MMLU_PROMPT = (
    "Answer this multiple-choice question. Respond with only the letter "
    "(A, B, C, or D) on the last line as 'Answer: <letter>'.\n\nQuestion: {q}\n"
)

def extract_number(text):
    text = text.replace(",", "")
    matches = re.findall(r"Answer:\s*(-?\d+\.?\d*)", text)
    if matches:
        return matches[-1]
    nums = re.findall(r"-?\d+\.?\d*", text)
    return nums[-1] if nums else None

def extract_letter(text):
    matches = re.findall(r"Answer:\s*([ABCD])", text.upper())
    if matches:
        return matches[-1]
    matches = re.findall(r"\b([ABCD])\b", text.upper())
    return matches[-1] if matches else None

@torch.no_grad()
def run_benchmark(generate_fn, questions, task_type, max_new_tokens=512, limit=None):
    qs = questions[:limit] if limit else questions
    correct = 0
    total_tokens = 0
    total_time = 0.0
    records = []

    for i, ex in enumerate(qs):
        prompt = GSM8K_PROMPT.format(q=ex["question"]) if task_type == "gsm8k" else MMLU_PROMPT.format(q=ex["question"])
        t0 = time.time()
        output_text, n_tokens = generate_fn(prompt, max_new_tokens)
        dt = time.time() - t0

        total_time += dt
        total_tokens += n_tokens

        pred = extract_number(output_text) if task_type == "gsm8k" else extract_letter(output_text)
        gold = ex["gold"]
        is_correct = (pred is not None) and (str(pred).strip() == str(gold).strip())
        correct += int(is_correct)

        records.append({
            "question": ex["question"][:80], "gold": gold, "pred": pred,
            "correct": is_correct, "subject": ex.get("subject"),
        })

        if (i + 1) % 20 == 0:
            print(f"  ...{i+1}/{len(qs)} done, running acc so far: {100*correct/(i+1):.1f}%")

    return {
        "accuracy_pct": 100 * correct / len(qs),
        "tokens_per_sec": total_tokens / total_time if total_time > 0 else 0,
        "n_questions": len(qs),
        "records": records,
    }
def measure_memory_gb():
    return torch.cuda.max_memory_allocated() / (1024**3)

def reset_memory():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

results_log = []
def log_result(variant_name, task_type, bench_result, peak_mem_gb, load_time_s):
    results_log.append({
        "variant": variant_name,
        "task": task_type,
        "accuracy_pct": bench_result["accuracy_pct"],
        "tokens_per_sec": bench_result["tokens_per_sec"],
        "peak_mem_gb": peak_mem_gb,
        "load_time_s": load_time_s,
        "n_questions": bench_result["n_questions"],
    })
    print(f"[{variant_name} | {task_type}] acc={bench_result['accuracy_pct']:.1f}%  "
          f"tok/s={bench_result['tokens_per_sec']:.1f}  mem={peak_mem_gb:.2f}GB")

In [ ]:
MODEL_ID = "microsoft/Phi-4-mini-instruct"

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from transformers import AutoModelForCausalLM

reset_memory()
t0 = time.time()
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="cuda"
)
load_time_fp16 = time.time() - t0

def make_generate_fn(model, tok):
    def generate_fn(prompt, max_new_tokens):
        inputs = tok(prompt, return_tensors="pt").to(model.device)
        in_len = inputs["input_ids"].shape[1]
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tok.pad_token_id,
        )
        n_new = out.shape[1] - in_len
        text = tok.decode(out[0][in_len:], skip_special_tokens=True)
        return text, n_new
    return generate_fn

gen_fp16 = make_generate_fn(model_fp16, tokenizer)


ValueError: `rope_scaling`'s short_factor field must have length 64, got 48

In [ ]:
res = run_benchmark(gen_fp16, gsm8k_questions, "gsm8k", limit=10)
print(res["accuracy_pct"], res["records"][:2])


10.0 [{'question': "Darrell and Allen's ages are in the ratio of 7:11. If their total age now is 162", 'gold': '109', 'pred': '109', 'correct': True, 'subject': None}, {'question': 'Lorraine and Colleen are trading stickers for buttons. Each large sticker is wor', 'gold': '89', 'pred': None, 'correct': False, 'subject': None}]


In [ ]:
def make_generate_fn(model, tok):
    def generate_fn(prompt, max_new_tokens):
        messages = [{"role": "user", "content": prompt}]
        inputs = tok.apply_chat_template(
            messages, add_generation_prompt=True,
            return_tensors="pt", return_dict=True
        ).to(model.device)
        in_len = inputs["input_ids"].shape[1]
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tok.pad_token_id,
        )
        n_new = out.shape[1] - in_len
        text = tok.decode(out[0][in_len:], skip_special_tokens=True)
        return text, n_new
    return generate_fn

gen_fp16 = make_generate_fn(model_fp16, tokenizer)

NameError: name 'model_fp16' is not defined

In [ ]:
reset_memory()

In [ ]:
prompt = GSM8K_PROMPT.format(q=gsm8k_questions[1]["question"])
text, n = gen_fp16(prompt, 512)
print(repr(text))

"Let's solve this step by step:\n\n1. Lorraine starts with 30 small stickers and 40 large stickers.\n\n2. She trades 90% of her small stickers for large buttons. \n   - 90% of 30 small stickers = 0.9 * 30 = 27 small stickers.\n   - Each large sticker is worth 3 small stickers, so 27 small stickers can be traded for 27 / 3 = 9 large buttons.\n\n3. She trades 50% of her large stickers for large buttons.\n   - 50% of 40 large stickers = 0.5 * 40 = 20 large stickers.\n   - Each large sticker is worth 1 large button, so 20 large stickers can be traded for 20 large buttons.\n\n4. She trades the rest of her large stickers for small buttons.\n   - She had 40 large stickers and traded 20, so she has 40 - 20 = 20 large stickers left.\n   - Each large sticker is worth 3 small stickers, so 20 large stickers can be traded for 20 * 3 = 60 small buttons.\n\n5. Now, let's calculate the total number of buttons Lorraine has:\n   - She has 9 large buttons from trading small stickers.\n   - She has 20 lar

In [ ]:
res = run_benchmark(gen_fp16, gsm8k_questions, "gsm8k", limit=10)
print(res["accuracy_pct"], res["records"][:2])

90.0 [{'question': "Darrell and Allen's ages are in the ratio of 7:11. If their total age now is 162", 'gold': '109', 'pred': '109', 'correct': True, 'subject': None}, {'question': 'Lorraine and Colleen are trading stickers for buttons. Each large sticker is wor', 'gold': '89', 'pred': '147', 'correct': False, 'subject': None}]


In [ ]:
res_gsm8k = run_benchmark(gen_fp16, gsm8k_questions, "gsm8k")
log_result("fp16", "gsm8k", res_gsm8k, measure_memory_gb(), load_time_fp16)

  ...20/200 done, running acc so far: 95.0%
  ...40/200 done, running acc so far: 82.5%
  ...60/200 done, running acc so far: 88.3%
  ...80/200 done, running acc so far: 88.8%
  ...100/200 done, running acc so far: 85.0%
  ...120/200 done, running acc so far: 86.7%
  ...140/200 done, running acc so far: 87.1%
  ...160/200 done, running acc so far: 87.5%
  ...180/200 done, running acc so far: 86.7%
  ...200/200 done, running acc so far: 86.5%
[fp16 | gsm8k] acc=86.5%  tok/s=19.5  mem=7.28GB


In [ ]:
res_mmlu = run_benchmark(gen_fp16, mmlu_questions, "mmlu")
log_result("fp16", "mmlu", res_mmlu, measure_memory_gb(), load_time_fp16)

  ...20/200 done, running acc so far: 50.0%
  ...40/200 done, running acc so far: 62.5%
  ...60/200 done, running acc so far: 63.3%
  ...80/200 done, running acc so far: 65.0%
  ...100/200 done, running acc so far: 63.0%
  ...120/200 done, running acc so far: 63.3%
  ...140/200 done, running acc so far: 64.3%
  ...160/200 done, running acc so far: 65.0%
  ...180/200 done, running acc so far: 65.6%
  ...200/200 done, running acc so far: 65.5%
[fp16 | mmlu] acc=65.5%  tok/s=17.9  mem=7.34GB


In [ ]:
res_mmlu = run_benchmark(gen_fp16, mmlu_questions, "mmlu")
log_result("fp16", "mmlu", res_mmlu, measure_memory_gb(), load_time_fp16)

  ...20/200 done, running acc so far: 50.0%
  ...40/200 done, running acc so far: 62.5%
  ...60/200 done, running acc so far: 63.3%
  ...80/200 done, running acc so far: 65.0%
  ...100/200 done, running acc so far: 63.0%
  ...120/200 done, running acc so far: 63.3%
  ...140/200 done, running acc so far: 64.3%
  ...160/200 done, running acc so far: 65.0%
  ...180/200 done, running acc so far: 65.6%
  ...200/200 done, running acc so far: 65.5%
[fp16 | mmlu] acc=65.5%  tok/s=17.8  mem=7.34GB


In [ ]:
from transformers import BitsAndBytesConfig

reset_memory()
t0 = time.time()
bnb_8bit_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
)
model_8bit = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_8bit_config, device_map="cuda",
    dtype=torch.float16,   # explicit, matches what generate_fn expects
)
load_time_8bit = time.time() - t0
gen_8bit = make_generate_fn(model_8bit, tokenizer)

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

In [ ]:
res = run_benchmark(gen_8bit, gsm8k_questions, "gsm8k", limit=10)
print(res["accuracy_pct"], res["records"][:2])

90.0 [{'question': "Darrell and Allen's ages are in the ratio of 7:11. If their total age now is 162", 'gold': '109', 'pred': '109', 'correct': True, 'subject': None}, {'question': 'Lorraine and Colleen are trading stickers for buttons. Each large sticker is wor', 'gold': '89', 'pred': '147', 'correct': False, 'subject': None}]


In [ ]:
try:
    del model_fp16
except NameError:
    pass
try:
    del gen_fp16
except NameError:
    pass
reset_memory()

In [ ]:
print(f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"Reserved: {torch.cuda.memory_reserved()/1024**3:.2f} GB")

Allocated: 4.16 GB
Reserved: 4.44 GB


In [ ]:
import gc
for obj in gc.get_objects():
    try:
        if torch.is_tensor(obj) or (hasattr(obj, 'data') and torch.is_tensor(obj.data)):
            pass  # just checking existence, not printing every tensor (too much output)
    except Exception:
        pass
print("model_8bit exists:", 'model_8bit' in dir())
print("model_fp16 exists:", 'model_fp16' in dir())

/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1164: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)
/tmp/ipykernel_690/2456530839.py:4: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if torch.is_tensor(obj) or (hasattr(obj, 'data') and torch.is_tensor(obj.data)):
/usr/local/lib/python3.12/dist-packages/torch/_inductor/ops_handler.py:789: UserWarning: undefined OpHandler.data, please add missing op schema
  warnings.warn(f"undefined OpHandler.{name}, please add missing op schema")


model_8bit exists: True
model_fp16 exists: False


In [ ]:
res_gsm8k = run_benchmark(gen_8bit, gsm8k_questions, "gsm8k")
log_result("bnb_8bit", "gsm8k", res_gsm8k, measure_memory_gb(), load_time_8bit)

res_mmlu = run_benchmark(gen_8bit, mmlu_questions, "mmlu")
log_result("bnb_8bit", "mmlu", res_mmlu, measure_memory_gb(), load_time_8bit)

del model_8bit, gen_8bit
reset_memory()

  ...20/200 done, running acc so far: 95.0%
  ...40/200 done, running acc so far: 87.5%
  ...60/200 done, running acc so far: 91.7%
  ...80/200 done, running acc so far: 90.0%
  ...100/200 done, running acc so far: 91.0%
  ...120/200 done, running acc so far: 90.8%
  ...140/200 done, running acc so far: 91.4%
  ...160/200 done, running acc so far: 91.2%
  ...180/200 done, running acc so far: 90.6%
  ...200/200 done, running acc so far: 90.0%
[bnb_8bit | gsm8k] acc=90.0%  tok/s=8.9  mem=4.28GB
  ...20/200 done, running acc so far: 45.0%
  ...40/200 done, running acc so far: 62.5%
  ...60/200 done, running acc so far: 63.3%
  ...80/200 done, running acc so far: 65.0%
  ...100/200 done, running acc so far: 63.0%
  ...120/200 done, running acc so far: 62.5%
  ...140/200 done, running acc so far: 63.6%
  ...160/200 done, running acc so far: 64.4%
  ...180/200 done, running acc so far: 65.0%
  ...200/200 done, running acc so far: 65.0%
[bnb_8bit | mmlu] acc=65.0%  tok/s=8.8  mem=4.36GB


In [ ]:
reset_memory()
t0 = time.time()
bnb_4bit_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model_nf4 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_4bit_config, device_map="cuda",
    dtype=torch.float16,
)
load_time_nf4 = time.time() - t0
gen_nf4 = make_generate_fn(model_nf4, tokenizer)

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

In [ ]:
res = run_benchmark(gen_nf4, gsm8k_questions, "gsm8k", limit=10)
print(res["accuracy_pct"], res["records"][:2])

80.0 [{'question': "Darrell and Allen's ages are in the ratio of 7:11. If their total age now is 162", 'gold': '109', 'pred': '109', 'correct': True, 'subject': None}, {'question': 'Lorraine and Colleen are trading stickers for buttons. Each large sticker is wor', 'gold': '89', 'pred': '69', 'correct': False, 'subject': None}]


In [ ]:
print(f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

Allocated: 4.25 GB


In [ ]:
print("model_8bit exists:", 'model_8bit' in dir())
print("model_nf4 exists:", 'model_nf4' in dir())


model_8bit exists: False
model_nf4 exists: True


In [ ]:
print(model_nf4.config.quantization_config)

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



In [ ]:
res_gsm8k = run_benchmark(gen_nf4, gsm8k_questions, "gsm8k")
log_result("bnb_nf4", "gsm8k", res_gsm8k, measure_memory_gb(), load_time_nf4)
gsm8k_nf4_records = res_gsm8k["records"]

res_mmlu = run_benchmark(gen_nf4, mmlu_questions, "mmlu")
log_result("bnb_nf4", "mmlu", res_mmlu, measure_memory_gb(), load_time_nf4)

del model_nf4, gen_nf4
reset_memory()


  ...20/200 done, running acc so far: 85.0%
  ...40/200 done, running acc so far: 85.0%
  ...60/200 done, running acc so far: 85.0%
  ...80/200 done, running acc so far: 85.0%
  ...100/200 done, running acc so far: 86.0%
  ...120/200 done, running acc so far: 86.7%
  ...140/200 done, running acc so far: 85.7%
  ...160/200 done, running acc so far: 86.2%
  ...180/200 done, running acc so far: 85.0%
  ...200/200 done, running acc so far: 85.0%
[bnb_nf4 | gsm8k] acc=85.0%  tok/s=15.2  mem=5.48GB
  ...20/200 done, running acc so far: 45.0%
  ...40/200 done, running acc so far: 55.0%
  ...60/200 done, running acc so far: 56.7%
  ...80/200 done, running acc so far: 60.0%
  ...100/200 done, running acc so far: 57.0%
  ...120/200 done, running acc so far: 57.5%
  ...140/200 done, running acc so far: 57.9%
  ...160/200 done, running acc so far: 58.1%
  ...180/200 done, running acc so far: 58.9%
  ...200/200 done, running acc so far: 58.0%
[bnb_nf4 | mmlu] acc=58.0%  tok/s=13.6  mem=5.48GB


In [ ]:
print("model_nf4 exists:", 'model_nf4' in dir())
print("model_8bit exists:", 'model_8bit' in dir())
print("model_fp16 exists:", 'model_fp16' in dir())
print(f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

model_nf4 exists: False
model_8bit exists: False
model_fp16 exists: False
Allocated: 0.01 GB


In [ ]:
!pip install -q autoawq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
!pip install -q "transformers==4.47.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 20.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import transformers
print(transformers.__version__)

4.47.1


In [ ]:
from awq import AutoAWQForCausalLM

AWQ_OUT_DIR = "./phi4mini-awq"

if not os.path.exists(AWQ_OUT_DIR):
    reset_memory()
    awq_model = AutoAWQForCausalLM.from_pretrained(MODEL_ID, safetensors=True)
    awq_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    quant_config = {"zero_point": True, "q_group_size": 128, "w_bit": 4, "version": "GEMM"}
    awq_model.quantize(awq_tokenizer, quant_config=quant_config)
    awq_model.save_quantized(AWQ_OUT_DIR)
    awq_tokenizer.save_pretrained(AWQ_OUT_DIR)
    del awq_model
    reset_memory()
else:
    print("AWQ checkpoint already built, reusing it.")

/usr/local/lib/python3.12/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
/usr/local/lib/python3.12/dist-packages/torch/jit/_script.py:1488: DeprecationWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.12/dist-package

configuration_phi3.py:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-4-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 21 files:   0%|          | 0/21 [00:00<?, ?it/s]

ImportError: cannot import name 'LossKwargs' from 'transformers.utils' (/usr/local/lib/python3.12/dist-packages/transformers/utils/__init__.py)